In [7]:
from langchain.text_splitter import CharacterTextSplitter
from langchain.document_loaders import AsyncChromiumLoader
from langchain.document_transformers import Html2TextTransformer
from langchain.vectorstores import FAISS
import nest_asyncio
from langchain.document_loaders import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings


loader = CSVLoader('/home/ubuntu/LLM/data/Tourist_attractions_ROI_formatted.csv')
docs = loader.load()



# Chunk text
text_splitter = CharacterTextSplitter(chunk_size=100, 
                                      chunk_overlap=0)
chunked_documents = text_splitter.split_documents(docs)



model_name = "/home/ubuntu/LLM/models/embedding/all-mpnet-base-v2"
model_kwargs = {'device': 'cuda'}
embedding_function = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs
)


# Load chunked documents into the FAISS index
db = FAISS.from_documents(chunked_documents, embedding_function)


# Connect query to FAISS index using a retriever
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 4}
)

In [ ]:
from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler

# Callbacks support token-wise streaming
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])
# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="../models/llm/mistral-7b-claude-chat.Q5_K_M.gguf",
    temperature=0.1,
    max_tokens=2000,
    n_ctx=4096,
    n_gpu_layers=35,
    top_p=1,
    callback_manager=callback_manager,
    verbose=True,  # Verbose is required to pass to the callback manager
)

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA A10G, compute capability 8.6, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA A10G) - 19994 MiB free
llama_model_loader: loaded meta data with 23 key-value pairs and 201 tensors from ../models/llm/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = tinyllama_tinyllama-1.1b-chat-v1.0
llama_model_loader: - kv   2:                       llama.context_length u32              = 2048
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   4:                          lla

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """[INST] You are a very experienced travel guide. You will suggest me places of similar type that are close to my first location. 
You are given a context with information about the touristic places in Belem. You must use this information to answer the user's question. Use only the context given to you to give your answer. If you do not know the answer, say "I don't know".

Context: {context}

Question: {question}

[/INST]
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [12]:
from langchain.chains import RetrievalQA

rag_pipeline = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=False,
    chain_type_kwargs={"prompt": prompt}
)

In [13]:
question = "What is the Postal Code of the Murutucu Ruins?"
result = rag_pipeline({"query": question})

/tmp/ipykernel_22853/883530895.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = rag_pipeline({"query": question})


and, andz, a,ment, thez.
mented.

llama_perf_context_print:        load time =    1844.58 ms
llama_perf_context_print: prompt eval time =    1844.44 ms /  2426 tokens (    0.76 ms per token,  1315.31 tokens per second)
llama_perf_context_print:        eval time =      64.17 ms /    16 runs   (    4.01 ms per token,   249.33 tokens per second)
llama_perf_context_print:       total time =    1921.96 ms /  2442 tokens


In [9]:
query = "What is the Postal Code of the Murutucu Ruins?"
docs = db.similarity_search(query)
print(docs[0].page_content)

address: Estr. da Ceasa - Curió-Utinga, Belém - PA, 66610-840, Brazil
categories/0: Tourist attraction
categories/1: 
categories/2: 
categories/3: 
categories/4: 
categories/5: 
categories/6: 
categoryName: Tourist attraction
cid: 1.6041E+19
city: Belém
countryCode: BR
fid: 0x92a48ced93485ab3:0xde9d3d6e85e54079
googleFoodUrl: 
locatedIn: 
location/lat: -1.4464368
location/lng: -48.4279047
menu: 
neighborhood: Curió-Utinga
phone: 
phoneUnformatted: 
placeId: ChIJs1pIk-2MpJIReUDlhW49nd4
postalCode: 66610-840
price: 
rank: 1
reserveTableUrl: 
reviewsCount: 169
reviewsDistribution/fiveStar: 90
reviewsDistribution/fourStar: 26
reviewsDistribution/oneStar: 13
reviewsDistribution/threeStar: 28
reviewsDistribution/twoStar: 12
state: State of Pará
street: Estr. da Ceasa
subTitle: Ruínas do Murutucu
title: Murutucu ruins
totalScore: 4
url: https://www.google.com/maps/search/?api=1&query=Murutucu%20ruins&query_place_id=ChIJs1pIk-2MpJIReUDlhW49nd4
website: 
addressFormatted: Estr. Da Ceasa - Curió

In [ ]:
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

text_generation_pipeline = transformers.pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    repetition_penalty=1.1,
    return_full_text=True,
    max_new_tokens=300,
)

prompt_template = """
### [INST] 
Instruction: Answer the question based on your 
fantasy football knowledge. Here is context to help:

{context}

### QUESTION:
{question} 

[/INST]
 """

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

# Create prompt from prompt template 
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

# Create llm chain 
llm_chain = LLMChain(llm=mistral_llm, prompt=prompt)